In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)
hf_path = "multimedia-synergy-lab/GuardChat"
full_path = "dataset/final_df.json"
train_path = "dataset/final_df_train.json"
test_path = "dataset/final_df_test.json"


In [ ]:
import json
import os
from datasets import Dataset, Features, Value, ClassLabel, Sequence
from huggingface_hub import login

# ====== LOGIN (nếu chưa login CLI) ======
login(token=os.environ.get("HF_TOKEN"))

# ====== FILES TO UPLOAD ======
split_to_path = {
    "train": train_path,
    "test": test_path,
    "full": full_path,
}


def flatten_conv(conv):
    return "\n".join([f"{t['role']}: {t['content']}" for t in conv])


# ====== LOAD ALL DATA FIRST (to infer labels globally) ======
all_records = []
raw_by_split = {}

for split, path in split_to_path.items():
    with open(path, "r", encoding="utf-8") as f:
        split_data = json.load(f)
    raw_by_split[split] = split_data
    all_records.extend(split_data)
    print(f"Loaded {split}: {len(split_data)} samples from {path}")


# ====== INFER LABELS FROM ALL FILES ======
category_names = sorted({d["category"] for d in all_records})
source_names = sorted({d["source"] for d in all_records})
conversation_generator_names = sorted(
    {d["conversation_generator"] for d in all_records})

print("Detected category:", category_names)
print("Detected source:", source_names)
print("Detected conversation_generator:", conversation_generator_names)


# ====== DEFINE FEATURES ======
features = Features({
    "id": Value("int32"),
    "category": ClassLabel(names=category_names),
    "prompt": Value("string"),
    "raw_prompt": Value("string"),
    "source": ClassLabel(names=source_names),
    "conversation_generator": ClassLabel(names=conversation_generator_names),
    "conversation": Sequence({
        "turn_id": Value("int32"),
        "role": ClassLabel(names=["user", "assistant"]),
        "content": Value("string"),
    }),
    "conversation_text": Value("string"),
})


# ====== BUILD + PUSH EACH SPLIT ======
for split, split_data in raw_by_split.items():
    for d in split_data:
        d["conversation_text"] = flatten_conv(d["conversation"])

    dataset_split = Dataset.from_list(split_data).cast(features)
    dataset_split.push_to_hub(
        hf_path,
        split=split,
        private=False,
    )
    print(f"Uploaded split '{split}' -> {hf_path}")


ImportError: The `notebook_login` function can only be used in a notebook (Jupyter or Colab) and you need the `ipywidgets` module: `pip install ipywidgets`.